In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from attribution import patching_effect
from dictionary_learning.trainers.top_k import AutoEncoderTopK
from nnsight import LanguageModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda:0"
model_name = "EleutherAI/pythia-70m-deduped"
model = LanguageModel(model_name, device_map=device, dispatch=True)

torch.manual_seed(0)

submodules = [model.gpt_neox.layers[2]]

ae_config = {
    "activation_dim": 512,
    "dict_size": 8192,
    "k": 30,
}
dictionaries = {submodule: AutoEncoderTopK(**ae_config).to(device) for submodule in submodules}

clean_prompt = "Hello, how are you?"


transformers_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float32,
        )

tokenizer = AutoTokenizer.from_pretrained(model_name)

input_ids = tokenizer(clean_prompt, return_tensors="pt").to(device)


def metric_fn(model):
    # last_token_logits = model.logits[:, -1, :]  # Shape: [batch_size, vocab_size]
    # return last_token_logits.sum()
    # acts = model.gpt_neox.layers[4].output[0]
    # return acts.sum()
    # Access logits through model's output
    logits = model.output.logits
    return logits.sum()


effects, nnsight_deltas, nnsight_grads, nnsight_total_effect = patching_effect(
    clean=input_ids,
    patch=None,
    model=model,
    submodules=submodules,
    dictionaries=dictionaries,
    metric_fn=metric_fn,
    metric_kwargs={},
    method="attrib",
    # steps=10,
)

print(effects)


In [ ]:
print(type(effects))
first_key = list(effects.keys())[0]
print(effects[first_key])
sparse_act = effects[first_key]

print(sparse_act.act.shape, sparse_act.resc.shape)
print(sparse_act.act.sum())
print(sparse_act.resc.sum())

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

transformers_model = model._model

# transformers_model.eval()

# Freeze parameters so we don't compute param grads
for param in transformers_model.parameters():
    # param.requires_grad_(False)
    param.requires_grad_(True)

activations = {}
# def save_activation_hook(module, input, output):
#     # Output is a tuple, get the hidden states (first element)
#     hidden_states = output[0]
#     # Force gradient tracking on hidden states
#     # hidden_states.requires_grad = True
#     hidden_states.retain_grad()
#     # Store the hidden states
#     activations[module] = hidden_states

sae = dictionaries[submodules[0]]
def save_activation_hook(module, input, output):
    # Unpack the hidden states from output if it’s a tuple
    x = output[0]
    # Encode + decode
    x_hat, feats = sae(x, output_features=True)
    residual = x - x_hat
    # Detach so the backward pass doesn't flow through the residual
    residual = residual.detach()
    # Recombine
    x_recon = x_hat + residual
    x_recon.retain_grad()
    activations[module] = x_recon

    # Return in the correct format for that submodule
    # e.g. if the submodule originally returned a tuple (x, something_else)
    return (x_recon,) + output[1:]


# Register a hook on multiple layers, e.g. layer 2, 5, 10
chosen_layers = [2]
handles = []
for layer_idx in chosen_layers:
    h = transformers_model.gpt_neox.layers[layer_idx].register_forward_hook(save_activation_hook)
    handles.append(h)

try:

    # Forward pass (this time with autograd ON for hidden states).
    outputs = transformers_model(**input_ids)
    loss = outputs.logits.sum()
    loss.backward()

    for layer_idx in chosen_layers:
        block = transformers_model.gpt_neox.layers[layer_idx]
        x_grad = activations[block].grad
        print(f"Gradient shape for layer {layer_idx}:", x_grad.shape)

finally:
    # Remove hooks to avoid side-effects if run repeatedly
    for h in handles:
        h.remove()


In [ ]:
sae = dictionaries[submodules[0]]
print(type(sae))
print(sae.b_dec.device)

In [ ]:
for layer_idx in chosen_layers:
        block = transformers_model.gpt_neox.layers[layer_idx]
        x_grad = activations[block].grad
        print(f"Gradient shape for layer {layer_idx}:", x_grad.shape)
        print(activations[block].shape)

# encoded_acts_BLF = sae.encode(activations[block])
# decoded_acts_BLD = sae.decode(encoded_acts_BLF)

decoded_acts_BLD, encoded_acts_BLF = sae(
                (activations[block] - 0.0), output_features=True
            )

resid = activations[block] - decoded_acts_BLD

decoder_DF = sae.decoder.weight.data.clone()
print(decoder_DF.shape)
print(resid.shape)

# x_grad: [B, S, D]
# encoded_acts_BSF: [B, S, F]
# decoder_DF: [D, F]  (each column is a dictionary vector)

# 1) Dot product between each position's gradient and *every* dictionary vector:
#    grad_x_dot_decoder[b,s,f] = x_grad[b,s,:] @ decoder[:,f]
grad_x_dot_decoder_BSF = torch.einsum("bsd,df->bsf", x_grad, decoder_DF)


# mean_grad = sum([f.act.grad for f in fs]) / steps
# mean_residual_grad = sum([f.res.grad for f in fs]) / steps
# grad = SparseAct(act=mean_grad, res=mean_residual_grad)

# 2) Multiply by code f_i at each position:
#    effect_BSF[b,s,f] = f_acts_BSF[b,s,f] * grad_x_dot_decoder[b,s,f]
effects_BSF = encoded_acts_BLF * grad_x_dot_decoder_BSF * -1

# 3) Sum if you want a single scalar or keep the shape if you want details:
effect_scalar = effects_BSF.sum()  # scalar

print(effects_BSF.shape)
print(effects_BSF.sum())

diff_BSF = effects_BSF - sparse_act.act
print(diff_BSF.sum(), diff_BSF.shape)
print(diff_BSF.mean(), diff_BSF.std(), diff_BSF.abs().mean(), diff_BSF.abs().max())

print(sparse_act.act.shape, sparse_act.resc.shape)
print(sparse_act.act.sum())

error_effects_BLD = resid * x_grad * -1
print(error_effects_BLD.sum())
print(sparse_act.resc.sum())

print(resid.shape, x_grad.shape, sparse_act.resc.shape, nnsight_deltas[submodules[0]].res.shape, nnsight_deltas[submodules[0]].act.shape, nnsight_grads[submodules[0]].res.shape, nnsight_grads[submodules[0]].act.shape)


In [ ]:
import einops
effects_F = einops.reduce(effects_BSF, "b s f -> f", "sum")
sparse_act_F = einops.reduce(sparse_act.act, "b s f -> f", "sum")

diff_F = effects_F - sparse_act_F

print(effects_F.max(), effects_F.min(), effects_F.mean(), effects_F.std())
print(sparse_act_F.max(), sparse_act_F.min(), sparse_act_F.mean(), sparse_act_F.std())

top_20_effects = effects_F.topk(20, dim=-1).indices
top_20_sparse = sparse_act_F.topk(20, dim=-1).indices

print(top_20_effects)
print(top_20_sparse)

print(diff_F.sum(), diff_F.shape)
print(diff_F.mean(), diff_F.std(), diff_F.abs().mean(), diff_F.abs().max())


In [ ]:
print(encoded_acts_BLF.sum())
print(nnsight_deltas[submodules[0]].act.sum())

print(grad_x_dot_decoder_BSF.sum())
print(nnsight_grads[submodules[0]].act.sum())
print(nnsight_grads[submodules[0]].act.shape)
print(nnsight_grads[submodules[0]].res.shape)
print(nnsight_grads[submodules[0]].res.sum())
print(x_grad.sum())

print(resid.sum())
print(nnsight_deltas[submodules[0]].res.sum())


In [ ]:
print(nnsight_deltas[submodules[0]].act.shape)
print(nnsight_grads[submodules[0]].act.shape)


In [ ]:
def compute_attributions(
    transformers_model: AutoModelForCausalLM,
    sae,
    input_ids,
    chosen_layers: list[int],
    loss_fn=lambda outputs: outputs.logits.sum(),
    use_stop_gradient: bool = False,
):
    """
    Runs a forward/backward pass on `transformers_model` using the given `sae` 
    for activation compression/decompression. Returns dictionaries keyed by 
    layer

    Args:
        transformers_model: A HuggingFace model (e.g., GPT-NeoX).
        sae: An AutoEncoderTopK instance (or similar) that implements 
             encode() and decode().
        input_ids: Model inputs, e.g., from tokenizer(..., return_tensors='pt').
        chosen_layers: List of layer indices to register forward hooks on.
        loss_fn: A function that takes model outputs and returns a scalar loss 
                 for the backward pass. Defaults to sum of logits.
    
    Returns:
        A dict of dicts. For each layer index in chosen_layers, we have:
            "grad_x_dot_decoder_BLF": Tensor
            "effects_BLF": Tensor
            "encoded_acts_BLF": Tensor
            "residual_BLD": Tensor
            "x_grad_BLD": Tensor
            "error_effects_BLD": Tensor
    """

    # Make sure gradients are enabled for model parameters
    for param in transformers_model.parameters():
        param.requires_grad_(True)

    # Dict to store the re-encoded layer outputs in the forward hook
    activations = {}

    def save_activation_hook(module, input, output):
        # Output is a tuple, get the hidden states (first element)
        hidden_states = output[0]
        # Force gradient tracking on hidden states
        # hidden_states.requires_grad = True
        hidden_states.retain_grad()
        # Store the hidden states
        activations[module] = hidden_states

    def stop_gradient_activation_hook(module, hook_input, hook_output):
        """
        Forward hook that encodes the hidden states x using `sae`,
        then reconstructs them with x_hat. The difference (residual) 
        is detached so it doesn't propagate gradient back through that part.
        """
        x = hook_output[0]  # hidden states
        x_hat, _ = sae(x, output_features=True)
        residual = x - x_hat
        # Detach residual so we don't accumulate its gradient
        residual = residual.detach()

        x_recon = x_hat + residual
        # Retain grad so we can do backward on it
        x_recon.retain_grad()

        activations[module] = x_recon
        
        # If the module originally returned a tuple, preserve the structure
        return (x_recon,) + hook_output[1:]

    # Register hooks
    handles = []
    for layer_idx in chosen_layers:
        layer_module = transformers_model.gpt_neox.layers[layer_idx]
        if use_stop_gradient:
            h = layer_module.register_forward_hook(stop_gradient_activation_hook)
        else:
            h = layer_module.register_forward_hook(save_activation_hook)
        handles.append(h)

    try:
        # Forward pass
        outputs = transformers_model(**input_ids)
        loss = loss_fn(outputs)
        loss.backward()

    finally:
        # Remove hooks to avoid side effects if function is called repeatedly
        for h in handles:
            h.remove()

    # Now gather the results
    results = {}
    decoder_weight_DF = sae.decoder.weight.data

    for layer_idx in chosen_layers:
        layer_module = transformers_model.gpt_neox.layers[layer_idx]
        # The post-hook activation
        x_BLD = activations[layer_module]
        x_grad_BLD = x_BLD.grad

        encoded_acts_BLF = sae.encode(x_BLD) 
        decoded_acts_BLD = sae.decode(encoded_acts_BLF)

        residual_BLD = x_BLD - decoded_acts_BLD

        grad_x_dot_decoder_BLF = torch.einsum(
            "bld,df->blf", x_grad_BLD, decoder_weight_DF
        )

        node_effects_BLF = encoded_acts_BLF * grad_x_dot_decoder_BLF * -1
        error_effects_BLD = residual_BLD * x_grad_BLD * -1

        # Store everything
        results[layer_idx] = {
            "grad_x_dot_decoder_BLF": grad_x_dot_decoder_BLF.detach(),
            "effects_BLF": node_effects_BLF.detach(),
            "encoded_acts_BLF": encoded_acts_BLF.detach(),
            "residual_BLD": residual_BLD.detach(),
            "x_grad_BLD": x_grad_BLD.detach(),
            "error_effects_BLD": error_effects_BLD.detach(),
        }

    return results

results = compute_attributions(
    transformers_model,
    sae,
    input_ids,
    chosen_layers,
    loss_fn=lambda outputs: outputs.logits.sum(),
    use_stop_gradient=True,
)

In [ ]:
print(sparse_act.act.shape, sparse_act.resc.shape)
print(sparse_act.act.sum(), sparse_act.resc.sum())

print(results[2]["effects_BLF"].sum())
print(results[2]["error_effects_BLD"].sum())